In [1]:
import sys
!{sys.executable} -m pip install earthengine-api



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import ee
ee.Authenticate(auth_mode='notebook')



Successfully saved authorization token.


In [10]:
import ee
import geemap

ee.Initialize(project='ee-iinee0804')
print("GEE 연결 성공!")


GEE 연결 성공!


In [1]:
import ee
import geemap
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# 1. GEE 초기화
ee.Initialize(project='bamti-489306')

# 2. UI 위젯 (싹 비우기 버튼 삭제)
start_date_widget = widgets.DatePicker(description='시작 날짜', value=datetime(2024, 6, 1))
end_date_widget = widgets.DatePicker(description='종료 날짜', value=datetime(2024, 6, 30))
update_button = widgets.Button(description='🚀 지도 업데이트', button_style='info')
export_button = widgets.Button(description='📊 AI 학습 데이터 추출', button_style='warning')
output = widgets.Output()

# 3. 지도 설정
Map = geemap.Map(center=[36.3, 127.8], zoom=7)
roi = ee.Geometry.Rectangle([124.5, 33.0, 131.0, 39.5])

current_data = {}

# 내부용 레이어 정리 함수 (사용자에게는 안 보임)
def _internal_clear():
    target_names = ["강수량", "NDVI", "지표면온도", "고도"]
    for layer in Map.layers:
        if layer.name in target_names:
            Map.remove_layer(layer)

def update_map(b=None):
    global current_data
    with output:
        output.clear_output()
        _internal_clear() # 업데이트 시 자동으로 이전 레이어 청소

        sd = start_date_widget.value.strftime('%Y-%m-%d')
        ed = end_date_widget.value.strftime('%Y-%m-%d')
        print(f"📡 {sd} ~ {ed} 데이터 갱신 중...")

        try:
            # 데이터 수집
            rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(sd, ed).mean().resample('bilinear').rename('Rain')
            s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate(sd, ed).filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)).median()
            ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
            lst = ee.ImageCollection("MODIS/061/MOD11A1").filterBounds(roi).filterDate(sd, ed).select('LST_Day_1km').mean().multiply(0.02).subtract(273.15).rename('LST')
            elev = ee.Image('USGS/SRTMGL1_003').clip(roi).rename('Elevation')

            current_data = {'rain': rain, 'ndvi': ndvi, 'lst': lst, 'elev': elev}

            # 레이어 추가
            Map.addLayer(rain.clip(roi), {"min": 0, "max": 15, "palette": ["white", "blue"]}, "강수량")
            Map.addLayer(ndvi.clip(roi), {"min": 0.2, "max": 0.8, "palette": ['yellow', 'green']}, "NDVI")
            Map.addLayer(lst.clip(roi), {"min": 15, "max": 40, "palette": ['blue', 'red']}, "지표면온도")
            Map.addLayer(elev, {"min": 0, "max": 2000, "palette": ['#f5f5f5', '#252525']}, "고도", False)

        except Exception as e:
            print(f"❌ 에러: {e}")

# (handle_interaction 및 export_data 함수는 그대로 유지)
def handle_interaction(**kwargs):
    if kwargs.get('type') == 'click':
        latlon = kwargs.get('coordinates')
        with output:
            output.clear_output()
            point = ee.Geometry.Point(latlon[::-1])
            years = [2020, 2021, 2022, 2023, 2024]
            results = []
            for y in years:
                img = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(point).filterDate(f"{y}-06-01", f"{y}-06-30").median()
                val = img.normalizedDifference(['B8', 'B4']).reduceRegion(ee.Reducer.mean(), point, 10).get('nd').getInfo()
                results.append(val)
            plt.figure(figsize=(6, 2)); plt.plot(years, results, marker='o', color='green'); plt.grid(True); plt.show()

def export_data(b):
    with output:
        output.clear_output()
        if not current_data: return
        center = ee.Geometry.Point([Map.center[1], Map.center[0]])
        combined = ee.Image([current_data['rain'], current_data['ndvi'], current_data['lst'], current_data['elev']])
        samples = combined.sample(region=center.buffer(5000).bounds(), scale=500, numPixels=30).getInfo()
        if 'features' in samples:
            df = pd.DataFrame([f['properties'] for f in samples['features']])
            display(df.head())

# 버튼 연결
update_button.on_click(update_map)
export_button.on_click(export_data)
Map.on_interaction(handle_interaction)

# 화면 표시
display(widgets.VBox([
    widgets.HBox([start_date_widget, end_date_widget, update_button, export_button]),
    output
]))
display(Map)
update_map()

Map(center=[36.3, 127.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…